# NASA C-MAPSS Turbofan Engine RUL Prediction
## Notebook 01: Exploratory Data Analysis & Sensor Dynamics

### Objective
In this notebook, we:
1. Load the NASA C-MAPSS FD001 dataset with correct column headers and zero data leakage.
2. Analyze engine lifetimes and run-to-failure cycle distributions.
3. Examine all 21 sensor channels and 3 operational settings.
4. Identify invariant (flat / zero-variance) sensors and explain the physical reason they occur.
5. Observe degradation trajectories of key health indicators across operating cycles.
6. Understand the correlation between sensor trends and Remaining Useful Life (RUL).

In [ ]:
import sys
from pathlib import Path

# Add project root to path for modular imports
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.config import ALL_COLUMNS, SENSOR_COLS, SETTING_COLS, SENSOR_INFO, FD001_DROP_SENSORS
from src.data_loader import load_dataset, get_dataset_summary

print("Libraries and configurations successfully loaded.")

### 1. Ingest Data and Verify Structure
We load the FD001 subset (100 training engines run to complete failure; 100 test engines truncated before failure).

In [ ]:
train_df, test_df, rul_true = load_dataset("FD001")
summary = get_dataset_summary(train_df, test_df, rul_true)

print("FD001 Summary Statistics:")
for k, v in summary.items():
    print(f"  {k:20s}: {v}")

train_df.head()

### 2. Engine Lifespan Distribution
Every engine in the training set starts with random initial manufacturing variation and wear, running until operational failure.

In [ ]:
engine_lifespans = train_df.groupby("unit_number")["time_cycles"].max()

plt.figure(figsize=(8, 4))
sns.histplot(engine_lifespans, bins=20, kde=True, color="#2b5c8f")
plt.axvline(engine_lifespans.mean(), color="red", linestyle="--", label=f"Mean: {engine_lifespans.mean():.1f} cycles")
plt.axvline(engine_lifespans.min(), color="green", linestyle=":", label=f"Min: {engine_lifespans.min()} cycles")
plt.axvline(engine_lifespans.max(), color="purple", linestyle=":", label=f"Max: {engine_lifespans.max()} cycles")
plt.title("Distribution of Engine Run-to-Failure Lifetimes (FD001)")
plt.xlabel("Cycles until Failure")
plt.ylabel("Count of Engines")
plt.legend()
plt.tight_layout()
plt.show()

### 3. Sensor Variance Analysis & Identifying Invariant Channels
In FD001, the simulation was run at a single sea-level operating condition. Consequently, ambient and demanded parameters do not vary.
We check the standard deviation of each sensor.

In [ ]:
stds = train_df[SENSOR_COLS].std()
flat_sensors = stds[stds < 1e-4].index.tolist()

print(f"Sensors with zero variance ({len(flat_sensors)} total):")
for s in flat_sensors:
    print(f"  {s:5s} : {SENSOR_INFO[s]["desc"]} (Std = {stds[s]:.6f})")

plt.figure(figsize=(10, 4))
colors = ["crimson" if s in flat_sensors else "steelblue" for s in SENSOR_COLS]
plt.bar(SENSOR_COLS, stds.values, color=colors)
plt.yscale("log")
plt.title("Sensor Standard Deviations (Log Scale)")
plt.xlabel("Sensor Channel")
plt.ylabel("Standard Deviation (log)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4. Sensor Degradation Trajectories Across Cycles
Here we visualize key active sensors for engines 1 through 5.
Notice how sensor values remain stable during initial cycles, and then exhibit progressive non-linear drift as High Pressure Compressor (HPC) wear intensifies.

In [ ]:
key_sensors = ["s_2", "s_3", "s_4", "s_7", "s_11", "s_12", "s_15", "s_20", "s_21"]
sample_units = [1, 2, 3, 4, 5]

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
axes = axes.flatten()

for idx, sensor in enumerate(key_sensors):
    ax = axes[idx]
    for unit in sample_units:
        subset = train_df[train_df["unit_number"] == unit]
        ax.plot(subset["time_cycles"], subset[sensor], label=f"Unit {unit}")
    ax.set_title(f"{sensor}: {SENSOR_INFO[sensor]["desc"]}", fontsize=10)
    ax.set_xlabel("Cycle")
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle("Degradation Trends for Engines 1-5 (FD001)", y=1.02)
plt.tight_layout()
plt.show()

### 5. Correlation with Remaining Useful Life (RUL)
We calculate actual run-to-failure RUL for training engines and evaluate Pearson correlation.

In [ ]:
train_copy = train_df.copy()
max_cycles = train_copy.groupby("unit_number")["time_cycles"].transform("max")
train_copy["RUL"] = max_cycles - train_copy["time_cycles"]

active_sensors = [s for s in SENSOR_COLS if s not in flat_sensors]
corrs = train_copy[active_sensors + ["RUL"]].corr()["RUL"].sort_values()

print("Sensor Correlation with RUL:")
for s, val in corrs.items():
    if s in SENSOR_INFO:
        print(f"  {s:5s} ({SENSOR_INFO[s]["desc"]:30s}): {val:+.4f}")

plt.figure(figsize=(10, 8))
sns.heatmap(train_copy[active_sensors + ["RUL"]].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation Matrix of Active Sensors with RUL")
plt.tight_layout()
plt.show()

### Key Findings & Engineering Takeaways for Viva Defense
1. **Zero-Variance Channels (6 sensors)**: `s_1`, `s_5`, `s_10`, `s_16`, `s_18`, `s_19` and `setting_3` are completely flat because FD001 represents a single constant flight condition (sea-level ambient temperature and pressure). Dropping them eliminates noise and prevents collinearity.
2. **Thermodynamic Drift**: High Pressure Compressor degradation causes temperatures (`s_2`, `s_3`, `s_4`) and back-pressure (`s_11`) to climb significantly as the controller increases fuel flow to meet target thrust.
3. **Coolant Bleed & Fuel Ratio**: Bleed flow rates (`s_20`, `s_21`) and pressure ratios (`s_7`) decrease monotonically as stage pressure drops.
4. **Initial Healthy Plateau**: During early cycles (first ~50-80 cycles), sensor values remain essentially flat in the healthy state before degradation accelerates. This physically justifies **piecewise-linear RUL capping** at ~125 cycles.